# Policy retrieval

Step 9 checks the RAG retriever: given a plain-language description of a suspicious transaction, does it return the right policies from the manual?

| Section | Question |
|---|---|
| Step 9c | What does the retriever return for one realistic case? |
| Step 9d | How often does the expected policy appear in the top 3? |

In [1]:
import pandas as pd
pd.set_option("display.max_colwidth", 80)

from sentinel.data_sources import POLICY_INDEX_DIR, POLICY_MANUAL_PATH
from sentinel.rag.index import retrieve_policies
from sentinel.rag.policies import load_policies

policies = load_policies(POLICY_MANUAL_PATH)
print("policies in manual:", len(policies))
print("index folder exists:", POLICY_INDEX_DIR.exists())

policies in manual: 12
index folder exists: True


## Steps 9a–9b: the policy manual and the index

`policies/policy_manual.md` holds 12 fictional policies, one `## POL-xxx` section each.
`load_policies()` splits the manual at those headings (one chunk per policy), and `scripts/build_policy_index.py` stores each chunk with its embedding in ChromaDB (`data/policy_index`, model all-MiniLM-L6-v2, cosine distance).

In [3]:
pd.DataFrame(
    [{"policy_id": p.policy_id, "title": p.title, "words": len(p.text.split())} for p in policies]
)

,policy_id,title,words
0,POL-001,Automatic block,38
1,POL-002,Investigation band,26
2,POL-003,Unusual amount for the customer,28
3,POL-004,High-value online purchase,26
4,POL-005,Rapid repeated activity,25
5,POL-006,Small test charge followed by large purchase,29
6,POL-007,Night-time high-value online spending,31
7,POL-008,High-risk categories,25
8,POL-009,No recent history,38
9,POL-010,Protected attributes,28


## Step 9c: first retrieval

A realistic investigation case, written the way the agent will describe it: amount, how unusual it is for the customer, time, category and recent activity.
The retriever should return the policies about unusual amounts, night-time online spending and rapid activity.

In [2]:
case = (
    "Online purchase of 390 USD in shopping_net at 23:14. "
    "The amount is 21 times the customer's 30-day average. "
    "Three earlier purchases on the same card in the last hour."
)

results = retrieve_policies(case, POLICY_INDEX_DIR, k=3)
pd.DataFrame(results)[["policy_id", "title", "distance"]]

,policy_id,title,distance
0,POL-004,High-value online purchase,0.336936
1,POL-007,Night-time high-value online spending,0.406272
2,POL-003,Unusual amount for the customer,0.423848


## Step 9d: retrieval quality

An evaluation set of 12 cases, each written so that one policy is clearly the right answer.
For each case we record where that policy ranks among the top 5 results.

**Metrics:** hit@1, hit@3 and hit@5, the share of cases where the expected policy appears in the top 1, 3 or 5.

In [4]:
eval_cases = [
    ("Fraud score is above the automatic block threshold.", "POL-001"),
    ("Score falls between the investigation and block thresholds; who makes the final decision?", "POL-002"),
    ("Purchase of 640 USD; the customer usually spends about 50 USD, 13 times their 30-day average.", "POL-003"),
    ("Online purchase of 1,450 USD; no purchase above 200 USD in the last month.", "POL-004"),
    ("Four transactions on the same card within 40 minutes.", "POL-005"),
    ("A 1.20 USD charge followed six hours later by a 900 USD purchase.", "POL-006"),
    ("Online purchase of 300 USD at 02:30 at night.", "POL-007"),
    ("Transaction in the misc_net category, which has an above-average fraud rate.", "POL-008"),
    ("The card has no transactions in the previous 30 days, so normal spending is unknown.", "POL-009"),
    ("Can the customer's age or gender be used when deciding whether to block?", "POL-010"),
    ("The transaction was held; how and when should the customer be contacted?", "POL-011"),
    ("The recommendation must cite the rules it relies on and the supporting evidence.", "POL-012"),
]

rows = []
for case, expected in eval_cases:
    found = [r["policy_id"] for r in retrieve_policies(case, POLICY_INDEX_DIR, k=5)]
    rank = found.index(expected) + 1 if expected in found else None
    rows.append({"expected": expected, "rank": rank, "top_3": found[:3]})

ranks = pd.DataFrame(rows)
print(ranks.to_string(index=False))
print()
for k in (1, 3, 5):
    print(f"hit@{k}: {(ranks['rank'] <= k).mean():.0%}")

expected  rank                       top_3
 POL-001     1 [POL-001, POL-008, POL-002]
 POL-002     1 [POL-002, POL-001, POL-010]
 POL-003     2 [POL-004, POL-003, POL-007]
 POL-004     1 [POL-004, POL-007, POL-006]
 POL-005     1 [POL-005, POL-006, POL-009]
 POL-006     1 [POL-006, POL-004, POL-003]
 POL-007     1 [POL-007, POL-004, POL-006]
 POL-008     1 [POL-008, POL-001, POL-003]
 POL-009     1 [POL-009, POL-006, POL-005]
 POL-010     1 [POL-010, POL-001, POL-011]
 POL-011     1 [POL-011, POL-003, POL-005]
 POL-012     1 [POL-012, POL-010, POL-002]

hit@1: 92%
hit@3: 100%
hit@5: 100%
